# 13a — Evaluation datasets and retrieval metrics — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/13.md) · [Course map](../PLAN.md) · [Project](../../projects/stage13/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Define labeled evaluation cases
- Calculate recall, precision, and reciprocal rank
- Separate component and end-to-end quality


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Designing an evaluation dataset

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

An evaluation case has a question, trusted access scope, expected behavior, supporting sources, category, and split. A golden label is the expected answer under a documented policy; it can still contain annotation errors and needs review.

Development cases guide changes. Locked holdout cases measure a selected system after those changes. If many questions share a source, split by source family when possible so near-duplicate evidence does not leak across sets.

Include answerable, unanswerable, forbidden, exact-number, multi-source, and dependency-failure cases. Different categories need different metrics. Retrieval recall cannot establish generated-answer support or correct abstention.

### Worked example: follow the values

A growth question needs prior and current statements and a formula. Record the expected fraction, units, both source IDs, and derivation before tuning retrieval. The same text asked by an unauthorized tenant needs a denial label with no private content, not the numerical answer.

### Pause and explain

Are sixty paraphrases of one statement equivalent to sixty independent source cases?

<details><summary>Check your reasoning after trying</summary>

No. They share evidence and failure modes. Record the source group and report both case count and group count.

</details>

### Common mistakes to check

Automatically generated labels are not automatically human-reviewed. Avoid repeatedly examining holdout results while tuning.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

An evaluation case needs an ID, user/tenant, question, relevant evidence IDs, expected facts, acceptable abstention, and category. Keep development cases separate from a final holdout. Deduplicate paraphrases by source family; otherwise near-identical questions exaggerate confidence. Store source and label versions and have a human review ambiguous cases.

Recall@k asks what fraction of all relevant documents appears in the top k. Precision@k asks what fraction of returned slots is relevant. Reciprocal rank is 1/rank of the first relevant result or zero when none is found; MRR averages it. State how empty relevance sets and fewer-than-k results are handled.

Answer quality needs correctness, relevance, format compliance, groundedness, citation support, and appropriate refusal. Agent quality also needs correct tools, arguments, bounded steps, and safe termination. Model judges are noisy and can be biased or manipulated by the evaluated answer; calibrate against blinded human ratings. A single overall score can hide permission failures or systematic weak slices.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
case = {"id": "q01", "tenant": "A", "question": "What was 2025 revenue?", "relevant_ids": ["A-2025"], "expected_value": 120}
print(case)


## Exercise 1: Recall at k

Use unique IDs in top k; return relevant-hit count / number of relevant IDs. An empty relevance set raises ValueError.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def recall_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    if not relevant or k < 1: raise ValueError("Invalid evaluation case")
    return len(set(retrieved[:k]) & relevant)/len(relevant)


In [ ]:
assert recall_at_k(["a", "a", "b"], {"a", "b"}, 2) == .5
assert recall_at_k([], {"a"}, 3) == 0
print("Exercise 1: checks passed")


**Why this works:** Use a separate abstention metric for questions with no relevant evidence.


## Exercise 2: Reciprocal rank

Return reciprocal rank of first relevant ID, else 0.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def reciprocal_rank(retrieved, relevant):
    return next((1/rank for rank, id in enumerate(retrieved, 1) if id in relevant), 0.)


In [ ]:
assert reciprocal_rank(["x", "a"], {"a"}) == .5
assert reciprocal_rank(["x"], {"a"}) == 0
print("Exercise 2: checks passed")


**Why this works:** MRR rewards finding one useful result early and does not measure complete coverage.


## Exercise 3: Citation precision

Given one boolean human support judgment per citation, return supported/total; no citations returns None.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def citation_precision(supports):
    return sum(supports)/len(supports) if supports else None


In [ ]:
assert citation_precision([True, False, True]) == 2/3
assert citation_precision([]) is None
print("Exercise 3: checks passed")


**Why this works:** A system returning no citations should not receive a perfect citation-support score by convention.


## Independent transfer

Label at least 40 development and 20 locked holdout questions before tuning the retriever. Grow to 100–500 curated cases over the capstone. Include permission denials, unsupported questions, multi-document comparisons, and exact numbers.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [13a interface and acceptance cases](../assignments/13a.md).
2. Copy the [blank starter](../assignments/starters/13a.py) to `work/assignments/13a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 13a --solution work/assignments/13a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/13a.md#13a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 13a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why component metrics?

   They locate whether an error came from retrieval, tool execution, or generation.

2. Can a judge replace all human review?

   No. Judges require calibration and targeted human audits, especially on high-impact failures.


## Reading and review

- [Primary reference 1](https://scikit-learn.org/stable/modules/model_evaluation.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
